<a href="https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/00_neural_networks_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Maritime Vision Applications
## Session 0 · Part 1 — Artificial Neural Networks
*Torben Globisch · Visual and Analytic Computing in Ocean Technologies (VACOT)*

---

| Notebook | Topic |
|---|---|
| **→ 00** | **Artificial Neural Networks: neurons, layers, learning** |
| 01 | Computer Vision & an MLP on MNIST |
| 02 | Convolutional Neural Networks |
| 03 | Vision Foundation Models: the CNN era |
| 04 | Vision Transformers |
| 05 | Self-supervised pretraining & transfer |

**By the end of this notebook you can**
1. write down what a single artificial neuron computes,
2. explain why we need non-linear activation functions,
3. describe the training loop: forward pass → loss → backpropagation → update,
4. train your first neural network in PyTorch (**Exercise 1**).

> 💡 **How to use this notebook:** run the cells top to bottom (`Shift + Enter`).
> Cells marked ✏️ are exercises. Cells titled **Solution** are hidden; double-click one to reveal it.
> Running a solution cell overwrites your own version, so skip it when you want to test your code.
> (Because the solutions are there, *Runtime → Run all* always works, which is handy before a lecture.)
> Animations come with a player: you can pause them and step through them frame by frame.

In [ ]:
# @title ⚙️ Setup: imports, colours & helpers — run me first { display-mode: "form" }
import math, io, time, urllib.request
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrowPatch
from IPython.display import HTML, display
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

# Colour palette — the same colours as the lecture slides
NAVY, BLUE, LAVENDER, ORANGE, GREEN, RED, GREY, LIGHT = (
    "#2D3A8C", "#5B67B8", "#A3A9DB", "#E8702A", "#3E9E64", "#C0392B", "#8A8FA8", "#EEF0F7")
plt.rcParams.update({
    "figure.dpi": 100, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlecolor": NAVY,
    "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, GREEN, BLUE, RED, GREY]),
    "animation.html": "jshtml", "animation.embed_limit": 80,
})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)
print(f"PyTorch {torch.__version__} | device: {device}")
if device.type == "cpu":
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU")

def play(anim):
    """Show a matplotlib animation as an HTML player (play / pause / step frame by frame)."""
    plt.close(anim._fig)
    return HTML(anim.to_jshtml())

# Maritime example photos from the COCO validation set (Flickr, Creative Commons licences).
MARITIME = {"trawler": 330369, "sailboat": 286422, "lifeboat": 400803, "marina": 442822,
            "fishing_boat": 451090, "yacht": 489014, "ferry": 543300}

def load_image(name, size=None):
    """Load one of the MARITIME photos (or any image URL) as an RGB PIL image."""
    url = f"http://images.cocodataset.org/val2017/{MARITIME[name]:012d}.jpg" if name in MARITIME else name
    with urllib.request.urlopen(url, timeout=30) as r:
        img = Image.open(io.BytesIO(r.read())).convert("RGB")
    if size is not None:  # centre-crop to a square, then resize
        s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
        img = img.crop((l, t, l + s, t + s)).resize((size, size), Image.BICUBIC)
    return img

---
# 1 · What is an Artificial Neural Network?

**Inspired by the brain.** The human brain has about 86 billion neurons connected by trillions of synapses.
A neuron collects signals through its *dendrites*, integrates them in the *soma* and, if the total
is strong enough, *fires* a signal down its *axon*.

An artificial neural network (ANN) copies this structure in a very simplified way:

| Biology | ANN |
|---|---|
| neuron | mathematical node |
| synapse strength | learnable **weight** $w$ |
| firing | **activation function** $\sigma$ |

> **Definition.** An ANN is a *parameterised function* $f_\theta : \mathbb{R}^n \to \mathbb{R}^m$, built from
> layers of neurons, that transforms an input into an output. *Learning* means finding good parameters $\theta$.

## 1.1 · The artificial neuron (Rosenblatt's perceptron, 1958)

Each neuron computes a **weighted sum** of its inputs, adds a **bias**, and applies an **activation function**:

$$ y = \sigma\Big(\sum_{i=1}^{n} w_i x_i + b\Big) = \sigma(\mathbf{w}^\top \mathbf{x} + b) $$

* $x_i$: input features   * $w_i$: weights (*learned*)   * $b$: bias (*learned*)   * $\sigma$: non-linear activation

In [ ]:
# @title 🎨 Figure: the artificial neuron { display-mode: "form" }
def draw_neuron(ax):
    ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")
    ys = [4.8, 3.0, 1.2]
    for i, y in enumerate(ys, 1):
        ax.add_patch(Circle((1, y), 0.45, fc=LAVENDER, ec=NAVY, lw=1.5))
        ax.text(1, y, f"$x_{i}$", ha="center", va="center", fontsize=13)
        ax.annotate("", xy=(4.45, 3.0), xytext=(1.45, y), arrowprops=dict(arrowstyle="-|>", color=NAVY, lw=1.4))
        ax.text(2.9, (y + 3.0) / 2 + 0.25, f"$w_{i}$", color=ORANGE, fontsize=12, ha="center")
    ax.add_patch(Circle((5, 3), 0.6, fc="white", ec=NAVY, lw=2)); ax.text(5, 3, r"$\Sigma$", ha="center", va="center", fontsize=16)
    ax.annotate("", xy=(5, 2.4), xytext=(5, 0.9), arrowprops=dict(arrowstyle="-|>", color=GREEN, lw=1.4))
    ax.text(5, 0.6, "$b$", color=GREEN, ha="center", fontsize=13)
    ax.add_patch(FancyBboxPatch((6.4, 2.45), 1.1, 1.1, boxstyle="round,pad=0.05", fc=ORANGE, ec=ORANGE, alpha=.9))
    ax.text(6.95, 3, r"$\sigma$", ha="center", va="center", fontsize=16, color="white")
    ax.annotate("", xy=(6.35, 3), xytext=(5.6, 3), arrowprops=dict(arrowstyle="-|>", color=NAVY, lw=1.4))
    ax.annotate("", xy=(9.0, 3), xytext=(7.6, 3), arrowprops=dict(arrowstyle="-|>", color=NAVY, lw=1.4))
    ax.text(9.3, 3, "$y$", fontsize=15, va="center")
    ax.text(5, 5.6, r"$y = \sigma(w_1x_1 + w_2x_2 + w_3x_3 + b)$", ha="center", fontsize=13, color=NAVY)

fig, ax = plt.subplots(figsize=(7, 4)); draw_neuron(ax); plt.show()

In code, a neuron is one line. Let's compute one by hand:

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])     # inputs
w = torch.tensor([0.5, -1.0, 0.25])   # weights
b = torch.tensor(0.1)                 # bias

z = w @ x + b                         # weighted sum: 0.5·1 − 1·2 + 0.25·3 + 0.1
y = torch.relu(z)                     # activation
print(f"z = {z.item():.2f}   →   y = relu(z) = {y.item():.2f}")

### 🕹️ Play with a neuron
A neuron with two inputs splits the input plane with a line: the line where $w_1x_1 + w_2x_2 + b = 0$.
Move the sliders and see how **weights rotate** the line, the **bias shifts** it, and the **activation**
changes what the output looks like.

In [ ]:
from ipywidgets import interact, FloatSlider, Dropdown

ACTIVATIONS = {
    "relu": torch.relu,
    "sigmoid": torch.sigmoid,
    "tanh": torch.tanh,
    "none (linear)": lambda z: z,
}

def neuron_playground(w1=1.0, w2=-0.5, b=0.0, activation="sigmoid"):
    g = torch.linspace(-3, 3, 200)
    X1, X2 = torch.meshgrid(g, g, indexing="xy")
    Y = ACTIVATIONS[activation](w1 * X1 + w2 * X2 + b)
    fig, ax = plt.subplots(figsize=(5, 4.2))
    im = ax.imshow(Y, extent=[-3, 3, -3, 3], origin="lower", cmap="coolwarm")
    ax.contour(X1, X2, w1 * X1 + w2 * X2 + b, levels=[0], colors=NAVY, linewidths=2)
    ax.set(xlabel="$x_1$", ylabel="$x_2$", title=f"y = {activation}({w1:.1f}·x₁ + {w2:.1f}·x₂ + {b:.1f})")
    plt.colorbar(im, ax=ax, label="neuron output y"); plt.show()

interact(neuron_playground,
         w1=FloatSlider(value=1.0, min=-3, max=3, step=0.1), w2=FloatSlider(value=-0.5, min=-3, max=3, step=0.1),
         b=FloatSlider(value=0.0, min=-3, max=3, step=0.1), activation=Dropdown(options=list(ACTIVATIONS)));

## 1.2 · Activation functions

| Name | Formula | Notes |
|---|---|---|
| **ReLU** | $\max(0, x)$ | default choice in modern networks: cheap, does not saturate for $x>0$ |
| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | squashes to $(0,1)$, useful for probabilities; gradient vanishes for large $\lvert x \rvert$ |
| **Tanh** | $\tanh(x)$ | squashes to $(-1,1)$, zero-centred |

The **derivative** matters as much as the function itself: it decides how much gradient flows back during training (Section 3).

In [ ]:
xs = torch.linspace(-5, 5, 400, requires_grad=True)
fig, axes = plt.subplots(2, 3, figsize=(11, 5), sharex=True)
for col, (name, fn, color) in enumerate([("ReLU", torch.relu, NAVY), ("Sigmoid", torch.sigmoid, ORANGE), ("Tanh", torch.tanh, GREEN)]):
    ys = fn(xs)
    grad, = torch.autograd.grad(ys.sum(), xs)          # derivative via autograd
    axes[0, col].plot(xs.detach(), ys.detach(), color=color, lw=2.5); axes[0, col].set_title(name)
    axes[1, col].plot(xs.detach(), grad, color=color, lw=2.5, ls="--")
    for ax in axes[:, col]: ax.axhline(0, color=GREY, lw=.6); ax.axvline(0, color=GREY, lw=.6)
axes[0, 0].set_ylabel("σ(x)"); axes[1, 0].set_ylabel("σ'(x)  (derivative)")
for ax in axes[1]: ax.set_xlabel("x")
plt.tight_layout(); plt.show()

## 1.3 · Why non-linearity?
Without $\sigma$, stacking layers is pointless: two linear layers $W_2(W_1x + b_1) + b_2$ are just **one** linear layer
$W x + b$ with $W = W_2W_1$ and $b = W_2b_1 + b_2$. Let's check this numerically:

In [ ]:
torch.manual_seed(0)
deep_linear = nn.Sequential(nn.Linear(2, 16), nn.Linear(16, 16), nn.Linear(16, 2))   # 3 layers, no activation

# Collapse the three layers into a single (W, b)
W = torch.eye(2); b = torch.zeros(2)
for layer in deep_linear:
    W, b = layer.weight @ W, layer.weight @ b + layer.bias

x = torch.randn(5, 2)
print("3-layer network output:\n", deep_linear(x).detach())
print("single linear map  W x + b:\n", (x @ W.T + b).detach())
print("identical:", torch.allclose(deep_linear(x), x @ W.T + b, atol=1e-6))

Same numbers. However deep you make it, a network **without** activations can only draw **straight** decision boundaries.
We will watch this happen in the demo in Section 4.

---
# 2 · How is an ANN built up?

An ANN consists of **stacked layers** of neurons:
* **Input layer**: the raw data (pixel values, sensor readings, …)
* **Hidden layers**: learn intermediate representations
* **Output layer**: the final prediction (e.g. one score per class)

In a **fully-connected (dense) layer** every neuron is connected to every neuron of the next layer.
A dense layer with $n_\text{in}$ inputs and $n_\text{out}$ outputs has $n_\text{in}\cdot n_\text{out}$ weights **plus** $n_\text{out}$ biases.

**Key hyperparameters** (chosen by *you*, not learned): depth (# layers) · width (neurons per layer) · activation function · learning rate $\eta$.

In [ ]:
# @title 🎨 Figure helper: draw a fully-connected network { display-mode: "form" }
def draw_mlp(ax, sizes, names=None, max_show=8):
    """Draw a dense network. Layers wider than `max_show` are drawn truncated with '⋮'."""
    ax.axis("off"); L = len(sizes); m = min(max(sizes), max_show)
    colors = [BLUE] + [LAVENDER] * (L - 2) + [ORANGE]
    pos = []
    for l, n in enumerate(sizes):
        shown = min(n, max_show)
        ys = np.linspace(-(shown - 1) / 2, (shown - 1) / 2, shown) if shown > 1 else np.array([0.0])
        pos.append([(l * 3.0, y) for y in ys])
    for l in range(L - 1):
        for (x0, y0) in pos[l]:
            for (x1, y1) in pos[l + 1]:
                ax.plot([x0, x1], [y0, y1], color=GREY, lw=0.5, alpha=0.6, zorder=1)
    for l, layer in enumerate(pos):
        for (x, y) in layer:
            ax.add_patch(Circle((x, y), 0.32, fc=colors[l], ec=NAVY, lw=1.2, zorder=2))
        if sizes[l] > max_show:
            ax.text(layer[0][0], 0, "⋮", ha="center", va="center", fontsize=16, zorder=3, color=NAVY,
                    bbox=dict(fc="white", ec="none"))
        label = (names[l] if names else ("Input" if l == 0 else "Output" if l == L - 1 else f"Hidden {l}"))
        ax.text(layer[0][0], -m / 2 - 0.3, f"{label}\nn = {sizes[l]}", ha="center", va="top", fontsize=10, color=NAVY)
    ax.set_xlim(-1, (L - 1) * 3 + 1); ax.set_ylim(-m / 2 - 1.8, m / 2 + 0.2); ax.set_aspect("equal")

fig, ax = plt.subplots(figsize=(6, 3.2)); draw_mlp(ax, [3, 4, 4, 2]); plt.show()

The same network in PyTorch. `nn.Sequential` simply chains layers:

In [ ]:
model = nn.Sequential(
    nn.Linear(3, 4), nn.ReLU(),   # input  → hidden 1
    nn.Linear(4, 4), nn.ReLU(),   # hidden 1 → hidden 2
    nn.Linear(4, 2),              # hidden 2 → output
)
print(model)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nparameters per layer:", [(n, p.numel()) for n, p in model.named_parameters()])
print("total:", count_params(model), "= (3·4+4) + (4·4+4) + (4·2+2)")

---
# 3 · How does an ANN learn?

Three steps, repeated for every **mini-batch** of training data:

1. **Forward pass**: the input $x$ flows layer by layer to a prediction $\hat y$.
2. **Loss computation**: compare the prediction $\hat y$ with the true label $y$.
   * Regression → *mean squared error* $\;\mathcal{L}_\text{MSE} = \frac{1}{N}\sum_i (y_i - \hat y_i)^2$
   * Classification → *cross-entropy* $\;\mathcal{L}_\text{CE} = -\sum_c y_c \log \hat y_c$
3. **Backpropagation + update**: the chain rule gives $\frac{\partial \mathcal{L}}{\partial w}$ for **every** weight; then
   **gradient descent** takes a small step downhill: $\;w \leftarrow w - \eta \, \frac{\partial \mathcal{L}}{\partial w}$

## 3.1 · Loss functions
Cross-entropy only looks at the probability assigned to the *correct* class, and punishes confident mistakes very hard:

In [ ]:
p = np.linspace(0.001, 1, 400)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(p, -np.log(p), color=ORANGE, lw=2.5)
axes[0].set(xlabel="predicted probability of the correct class  $\\hat y_c$", ylabel="$-\\log \\hat y_c$",
            title="Cross-entropy (classification)", ylim=(0, 6))
axes[0].annotate("confident & wrong\n→ huge loss", xy=(0.02, 3.9), xytext=(0.25, 4.7), arrowprops=dict(arrowstyle="->", color=NAVY))
axes[0].annotate("confident & right\n→ loss ≈ 0", xy=(0.97, 0.05), xytext=(0.6, 1.6), arrowprops=dict(arrowstyle="->", color=NAVY))
e = np.linspace(-3, 3, 400)
axes[1].plot(e, e**2, color=NAVY, lw=2.5)
axes[1].set(xlabel="error  $y - \\hat y$", ylabel="$(y-\\hat y)^2$", title="Squared error (regression)")
plt.tight_layout(); plt.show()

## 3.2 · Gradient descent: walking downhill
Think of the loss as a landscape over the weights. The gradient points uphill, so we step in the **opposite**
direction. The **learning rate** $\eta$ sets the step size, and it matters a lot:

In [ ]:
# A simple 2-D "loss landscape" over two weights (an elongated bowl, minimum at (0, 0))
A = np.array([[1.0, 0.0], [0.0, 8.0]])
R = np.array([[np.cos(0.5), -np.sin(0.5)], [np.sin(0.5), np.cos(0.5)]]); A = R @ A @ R.T
loss_fn = lambda w: 0.5 * w @ A @ w
grad_fn = lambda w: A @ w

def gradient_descent(lr, steps=40, w0=(-2.6, 1.6)):
    w = np.array(w0); path = [w.copy()]
    for _ in range(steps):
        w = w - lr * grad_fn(w)          # ← the whole update rule
        path.append(w.copy())
    return np.array(path)

runs = {"η = 0.02  (too small)": (gradient_descent(0.02), GREEN),
        "η = 0.12  (good)": (gradient_descent(0.12), NAVY),
        "η = 0.255 (too large)": (gradient_descent(0.255), RED)}

g = np.linspace(-3, 3, 200); G1, G2 = np.meshgrid(g, g)
Z = 0.5 * (A[0, 0] * G1**2 + 2 * A[0, 1] * G1 * G2 + A[1, 1] * G2**2)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), gridspec_kw=dict(width_ratios=[1, 1, 1, 1.2]))
lines, dots, curves = [], [], []
for ax, (name, (path, c)) in zip(axes[:3], runs.items()):
    ax.contourf(G1, G2, Z, levels=25, cmap="Blues_r", alpha=.85); ax.contour(G1, G2, Z, levels=12, colors="white", linewidths=.4)
    ax.plot(0, 0, "*", color="gold", ms=14, mec="k"); ax.set_title(name, color=c); ax.set_aspect("equal")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel("$w_1$"); ax.set_ylabel("$w_2$")
    lines.append(ax.plot([], [], "-o", color=c, ms=3, lw=1.5, mfc="white")[0])
for name, (path, c) in runs.items():
    curves.append(axes[3].plot([], [], color=c, lw=2, label=name.split("(")[0])[0])
axes[3].set(yscale="log", xlim=(0, 40), ylim=(1e-4, 1e4), xlabel="step", ylabel="loss", title="Loss during descent"); axes[3].legend(fontsize=8)
plt.tight_layout()

def update(t):
    for ln, cv, (path, c) in zip(lines, curves, runs.values()):
        ln.set_data(path[:t + 1, 0], path[:t + 1, 1])
        cv.set_data(range(t + 1), [loss_fn(w) for w in path[:t + 1]])
    return lines + curves

play(animation.FuncAnimation(fig, update, frames=41, interval=150))

* too small → slow, still far from the minimum after 40 steps
* good → fast convergence
* too large → overshoots, bounces from wall to wall and **diverges** (the loss explodes)

For this bowl the threshold is exactly $\eta = 2/\lambda_\text{max} = 0.25$, where $\lambda_\text{max}=8$ is the steepest curvature.
Real loss landscapes are not bowls, but the lesson carries over: the steepest direction limits the learning rate.

## 3.3 · Backpropagation = the chain rule
For a single neuron $\hat y = \sigma(wx + b)$ with loss $\mathcal{L} = (\hat y - y)^2$ the chain rule gives

$$\frac{\partial \mathcal{L}}{\partial w} = \underbrace{2(\hat y - y)}_{\partial \mathcal{L}/\partial \hat y}\cdot\underbrace{\sigma'(z)}_{\partial \hat y/\partial z}\cdot\underbrace{x}_{\partial z/\partial w}, \qquad z = wx+b$$

Backpropagation applies this rule layer by layer, from the output back to the input. PyTorch's **autograd** does it for us;
let's check that it agrees with the formula above:

In [ ]:
x, y = torch.tensor(1.5), torch.tensor(1.0)
w = torch.tensor(0.8, requires_grad=True)
b = torch.tensor(-0.3, requires_grad=True)

z = w * x + b
y_hat = torch.sigmoid(z)
loss = (y_hat - y) ** 2
loss.backward()                                           # ← backpropagation

sig = torch.sigmoid(z).detach()
by_hand = 2 * (y_hat.detach() - y) * sig * (1 - sig) * x  # σ'(z) = σ(z)(1 − σ(z))
print(f"dL/dw  autograd: {w.grad.item():.6f}   by hand: {by_hand.item():.6f}")

## 3.4 · The training loop in PyTorch
Every training script you will ever see has this skeleton:

```python
for epoch in range(num_epochs):
    for x, y in train_loader:            # one mini-batch
        y_hat = model(x)                 # 1. forward pass
        loss  = loss_fn(y_hat, y)        # 2. loss
        optimizer.zero_grad()            #    (reset old gradients)
        loss.backward()                  # 3a. backpropagation
        optimizer.step()                 # 3b. w ← w − η ∂L/∂w
```

---
# 4 · Demo: a small ANN in action
Let's put everything together on a classic toy problem: two interleaving half-moons. A straight line **cannot** separate them.
We train two networks with the *same* architecture (2 → 16 → 16 → 2):
* **left:** without activation functions (a "deep" linear model)
* **right:** with ReLU activations

In [ ]:
from sklearn.datasets import make_moons
Xm, ym = make_moons(n_samples=400, noise=0.2, random_state=0)
Xm, ym = torch.tensor(Xm, dtype=torch.float32), torch.tensor(ym)

def make_net(activation):
    act = activation if activation is not None else nn.Identity()
    return nn.Sequential(nn.Linear(2, 16), act, nn.Linear(16, 16), act, nn.Linear(16, 2))

torch.manual_seed(1)
nets = {"no activation": make_net(None), "ReLU": make_net(nn.ReLU())}
gx, gy = torch.meshgrid(torch.linspace(-2, 3, 150), torch.linspace(-1.5, 2, 150), indexing="xy")
grid = torch.stack([gx.ravel(), gy.ravel()], 1)

snapshots = {k: [] for k in nets}; losses = {k: [] for k in nets}
for name, net in nets.items():
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    for epoch in range(301):
        loss = F.cross_entropy(net(Xm), ym)          # forward + loss (full batch)
        opt.zero_grad(); loss.backward(); opt.step()  # backprop + update
        losses[name].append(loss.item())
        if epoch % 10 == 0:
            with torch.no_grad():
                snapshots[name].append(net(grid).softmax(1)[:, 1].reshape(gx.shape))
    acc = (net(Xm).argmax(1) == ym).float().mean()
    print(f"{name:>14}: final accuracy {acc:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
ims = []
for ax, name in zip(axes[:2], nets):
    ims.append(ax.imshow(snapshots[name][0], extent=[-2, 3, -1.5, 2], origin="lower", cmap="coolwarm", vmin=0, vmax=1, alpha=.7))
    ax.scatter(*Xm[ym == 0].T, c=BLUE, s=12, ec="white", lw=.4); ax.scatter(*Xm[ym == 1].T, c=ORANGE, s=12, ec="white", lw=.4)
    ax.set_title(f"2→16→16→2, {name}"); ax.set_xticks([]); ax.set_yticks([])
for name, c in zip(nets, [GREY, NAVY]):
    axes[2].plot(losses[name], color=c, lw=.8, alpha=.25)
curves = [axes[2].plot([], [], color=c, lw=2.5, label=n)[0] for n, c in zip(nets, [GREY, NAVY])]
axes[2].set(xlabel="epoch", ylabel="cross-entropy loss", title="Training loss", xlim=(0, 300)); axes[2].legend()
title = fig.suptitle("", color=NAVY, fontweight="bold"); plt.tight_layout()

def update(i):
    for im, name in zip(ims, nets): im.set_data(snapshots[name][i])
    for cv, name in zip(curves, nets): cv.set_data(range(i * 10 + 1), losses[name][:i * 10 + 1])
    title.set_text(f"epoch {i * 10}")
    return ims + curves

play(animation.FuncAnimation(fig, update, frames=len(snapshots["ReLU"]), interval=120))

The linear model is stuck with a straight boundary however long we train. With ReLU the network **bends** the
boundary around the data. That is exactly why every hidden layer has a non-linearity.

---
# ✏️ Exercise 1: Classify flowers with a neural network

**Dataset: Iris** (ships with scikit-learn)
* 150 samples, 3 flower classes: *setosa*, *versicolor*, *virginica*
* 4 features: sepal length, sepal width, petal length, petal width (cm)
* split: 80 % train / 20 % test

**Network:** 4 → 8 → 8 → 3 with ReLU, trained with cross-entropy.
Total trainable parameters: $(4\cdot8+8) + (8\cdot8+8) + (8\cdot3+3) = 139$.

🎯 **Goal:** test accuracy > 90 %

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4)); draw_mlp(ax, [4, 8, 8, 3], ["Input\n(4 features)", "Hidden 1", "Hidden 2", "Output\n(3 classes)"]); plt.show()

### Step 0: look at the data
Always look at your data before modelling it!

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X_np, y_np = iris.data.astype(np.float32), iris.target
CLASS_NAMES = list(iris.target_names)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for c, color in enumerate([BLUE, ORANGE, GREEN]):
    axes[0].scatter(*X_np[y_np == c][:, [0, 1]].T, color=color, label=CLASS_NAMES[c], s=22, ec="white", lw=.5)
    axes[1].scatter(*X_np[y_np == c][:, [2, 3]].T, color=color, label=CLASS_NAMES[c], s=22, ec="white", lw=.5)
axes[0].set(xlabel="sepal length (cm)", ylabel="sepal width (cm)", title="Sepals")
axes[1].set(xlabel="petal length (cm)", ylabel="petal width (cm)", title="Petals"); axes[1].legend()
plt.tight_layout(); plt.show()

Petals separate the classes much better than sepals. Next we split the data and **standardise** each feature
(zero mean, unit variance). Neural networks train much more smoothly on standardised inputs.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, stratify=y_np, random_state=42)
mean, std = X_train.mean(0), X_train.std(0)          # statistics from the TRAIN set only!
X_train, X_test = (X_train - mean) / std, (X_test - mean) / std

X_train, X_test = torch.tensor(X_train), torch.tensor(X_test)
y_train, y_test = torch.tensor(y_train), torch.tensor(y_test)
print("train:", tuple(X_train.shape), " test:", tuple(X_test.shape))

### ✏️ Step 1: build the network
Fill in `self.net` with an `nn.Sequential`: **4 → 8 → 8 → 3** with `nn.ReLU()` between the linear layers.
(No softmax at the end: `F.cross_entropy` expects raw scores, called *logits*.)

In [ ]:
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: replace None with nn.Sequential(...)
        self.net = None

    def forward(self, x):
        return self.net(x)

In [ ]:
# @title 💡 Solution: Step 1 { display-mode: "form" }
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 8), nn.ReLU(),
            nn.Linear(8, 8), nn.ReLU(),
            nn.Linear(8, 3),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
model = IrisNet()
print(model)
assert count_params(model) == 139, f"Expected 139 parameters, got {count_params(model)}. Check your layer sizes!"
print("✅ 139 trainable parameters, as computed on the slide")

### ✏️ Step 2: write the training loop
Implement the four lines inside the loop (see Section 3.4). The data is tiny, so we use the whole training set as one batch.

In [ ]:
def train_iris(model, epochs=150, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train": [], "test": [], "test_acc": []}
    for epoch in range(epochs):
        model.train()
        # TODO 1: forward pass   → logits = ...
        # TODO 2: loss           → loss = F.cross_entropy(...)
        # TODO 3: backprop       → zero_grad, backward
        # TODO 4: update         → optimizer.step()
        loss = torch.tensor(float("nan"))  # remove this line once you are done

        model.eval()
        with torch.no_grad():
            test_logits = model(X_test)
            history["train"].append(loss.item())
            history["test"].append(F.cross_entropy(test_logits, y_test).item())
            history["test_acc"].append((test_logits.argmax(1) == y_test).float().mean().item())
    return history

In [ ]:
# @title 💡 Solution: Step 2 { display-mode: "form" }
def train_iris(model, epochs=150, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train": [], "test": [], "test_acc": []}
    for epoch in range(epochs):
        model.train()
        logits = model(X_train)                   # 1. forward pass
        loss = F.cross_entropy(logits, y_train)   # 2. loss
        optimizer.zero_grad()
        loss.backward()                           # 3. backpropagation
        optimizer.step()                          # 4. update

        model.eval()
        with torch.no_grad():
            test_logits = model(X_test)
            history["train"].append(loss.item())
            history["test"].append(F.cross_entropy(test_logits, y_test).item())
            history["test_acc"].append((test_logits.argmax(1) == y_test).float().mean().item())
    return history

### Step 3: train & evaluate

In [ ]:
torch.manual_seed(0)
model = IrisNet()
history = train_iris(model, epochs=150, lr=0.01)
print(f"final test accuracy: {history['test_acc'][-1]:.1%}")

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from matplotlib.colors import LinearSegmentedColormap
NAVY_CMAP = LinearSegmentedColormap.from_list("navy", ["white", NAVY])

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].plot(history["train"], label="train loss", lw=2); axes[0].plot(history["test"], label="test loss", lw=2)
axes[0].set(xlabel="epoch", ylabel="loss", title="Loss curves"); axes[0].legend()
axes[1].plot(history["test_acc"], color=GREEN, lw=2); axes[1].axhline(0.9, color=RED, ls="--", lw=1, label="goal: 90 %")
axes[1].set(xlabel="epoch", ylabel="accuracy", title="Test accuracy", ylim=(0, 1.02)); axes[1].legend()
with torch.no_grad(): pred = model(X_test).argmax(1)
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=CLASS_NAMES).plot(ax=axes[2], cmap=NAVY_CMAP, colorbar=False)
axes[2].set_title("Confusion matrix (test)")
plt.tight_layout(); plt.show()

### 🔬 Experiments (pick one or two)
1. **Learning rate:** retrain with `lr=1.0` and `lr=0.0001`. What happens to the loss curve? (Think back to the gradient-descent animation.)
2. **No hidden layers:** replace the network with a single `nn.Linear(4, 3)`. Is Iris linearly separable?
3. **No standardisation:** comment out the standardisation line and retrain. What changes?
4. **Overfitting:** make the network much bigger (e.g. 4 → 256 → 256 → 3) and train for 1000 epochs. Watch train vs. test loss.

In [ ]:
# Your experiments here
torch.manual_seed(0)
h_big_lr = train_iris(IrisNet(), epochs=150, lr=1.0)
h_small_lr = train_iris(IrisNet(), epochs=150, lr=0.0001)
plt.figure(figsize=(6, 3.5))
for h, lbl in [(history, "lr = 0.01"), (h_big_lr, "lr = 1.0"), (h_small_lr, "lr = 0.0001")]:
    plt.plot(h["train"], label=lbl, lw=2)
plt.xlabel("epoch"); plt.ylabel("train loss"); plt.legend(); plt.title("Effect of the learning rate"); plt.show()

---
# Summary
1. A **neuron** computes a weighted sum plus a bias, followed by an activation: $y = \sigma(\mathbf{w}^\top\mathbf{x} + b)$.
2. **Non-linear activations** are essential; without them a deep network collapses to a single linear map.
3. An **ANN** stacks layers of neurons; a dense layer has $n_\text{in} n_\text{out} + n_\text{out}$ parameters.
4. Networks **learn** by repeating: forward pass → loss → backpropagation (chain rule) → gradient-descent update.
5. The **learning rate** decides whether training crawls, converges or explodes.

**Next →** [01 · Computer Vision & an MLP on MNIST](https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/01_computer_vision_and_mlp_mnist.ipynb): what happens when the input is an image?